# Phase 1 — Segmentation
## Mask IoU 원리 + SAM2 추론 + mIoU 평가 파이프라인

### 학습 목표
- **Mask IoU / Dice** — 세그멘테이션 평가 지표를 Detection IoU와 비교하여 직접 구현
- **SAM2 아키텍처** — Image Encoder(ViT), Prompt Encoder, Mask Decoder 원리 이해
- **프롬프트 방식** — Point / Box / Automatic 세 가지 방식 비교
- **mIoU 평가 파이프라인** — Box prompt → Mask 생성 → IoU 집계까지 직접 구현
- **자율주행 응용** — 차량/보행자/도로 세그멘테이션 분석

### Phase 1 Detection과의 연결
```
Detection  : bbox → IoU(box)  → mAP
Segmentation: mask → IoU(mask) → mIoU
```
구조는 동일, 평가 단위만 box → mask 로 바뀜

## 0. 환경 확인

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import torch
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')
print(f'NumPy   : {np.__version__}')
print(f'OpenCV  : {cv2.__version__}')

PROJECT_ROOT = Path('C:/Users/apple/Desktop/autonomous_cv_pipeline')
COCO_IMG_DIR = Path.home() / 'MyProject' / 'datasets' / 'coco128' / 'images' / 'train2017'
COCO_LBL_DIR = Path.home() / 'MyProject' / 'datasets' / 'coco128' / 'labels' / 'train2017'
print(f'\nCOCO128 이미지: {len(list(COCO_IMG_DIR.glob("*.jpg")))}장')

## 1. Mask IoU / Dice / Precision@Mask 직접 구현

### Detection IoU vs Segmentation IoU
```
Detection  IoU: 두 사각형의 겹치는 넓이 / 합친 넓이
Mask       IoU: 두 이진 마스크의 픽셀 단위 교집합 / 합집합
```

### 세그멘테이션 주요 지표
| 지표 | 수식 | 특징 |
|------|------|------|
| **IoU (Jaccard)** | TP / (TP+FP+FN) | 0~1, 표준 평가 지표 |
| **Dice (F1)** | 2·TP / (2·TP+FP+FN) | IoU보다 큰 값, 의료 영상에 많이 사용 |
| **Pixel Accuracy** | (TP+TN) / Total | 클래스 불균형에 취약 |
| **mIoU** | 클래스별 IoU 평균 | Detection mAP와 대응 |

In [ ]:
def mask_iou(pred_mask: np.ndarray, gt_mask: np.ndarray) -> float:
    """
    이진 마스크 IoU (Intersection over Union).
    pred_mask, gt_mask: bool 또는 0/1 ndarray, 같은 shape
    """
    pred = pred_mask.astype(bool)
    gt   = gt_mask.astype(bool)
    intersection = np.logical_and(pred, gt).sum()
    union        = np.logical_or(pred, gt).sum()
    return float(intersection) / float(union + 1e-8)


def mask_dice(pred_mask: np.ndarray, gt_mask: np.ndarray) -> float:
    """
    Dice Coefficient (F1 Score for masks).
    Dice = 2·|A∩B| / (|A|+|B|)
    """
    pred = pred_mask.astype(bool)
    gt   = gt_mask.astype(bool)
    intersection = np.logical_and(pred, gt).sum()
    return 2.0 * intersection / (pred.sum() + gt.sum() + 1e-8)


def mask_precision_recall(pred_mask: np.ndarray, gt_mask: np.ndarray):
    """
    픽셀 단위 Precision / Recall / F1
    """
    pred = pred_mask.astype(bool)
    gt   = gt_mask.astype(bool)
    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum()
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    return float(precision), float(recall), float(f1)


# ── 검증: 합성 마스크로 직접 계산 ──
H, W = 100, 100
gt_mask   = np.zeros((H, W), dtype=bool)
pred_mask = np.zeros((H, W), dtype=bool)

gt_mask[20:80, 20:80]   = True   # 60×60 = 3600픽셀
pred_mask[40:90, 40:90] = True   # 50×50 = 2500픽셀

# 교집합: [40:80, 40:80] = 40×40 = 1600
# 합집합: 3600 + 2500 - 1600 = 4500
iou  = mask_iou(pred_mask, gt_mask)
dice = mask_dice(pred_mask, gt_mask)
prec, rec, f1 = mask_precision_recall(pred_mask, gt_mask)

print(f'Mask IoU        : {iou:.4f}  (기대값: {1600/4500:.4f})')
print(f'Dice Coefficient: {dice:.4f}  (기대값: {2*1600/(3600+2500):.4f})')
print(f'Precision       : {prec:.4f}')
print(f'Recall          : {rec:.4f}')
print(f'F1              : {f1:.4f}')
print()
print('[관계] Dice = 2·IoU / (1+IoU) ← IoU > Dice 항상 성립')
print(f'  검증: 2·{iou:.4f}/(1+{iou:.4f}) = {2*iou/(1+iou):.4f} (Dice={dice:.4f})')

In [ ]:
# ── IoU / Dice 관계 시각화 ──
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 마스크 시각화
vis = np.zeros((H, W, 3), dtype=np.uint8)
vis[gt_mask]                          = [100, 200, 100]   # GT: 초록
vis[pred_mask & ~gt_mask]             = [200, 100, 100]   # FP: 빨강
vis[np.logical_and(gt_mask, pred_mask)] = [200, 200, 100]  # TP: 노랑

axes[0].imshow(vis)
axes[0].set_title('마스크 겹침 구조')
patches = [
    mpatches.Patch(color=[100/255,200/255,100/255], label='GT only (FN)'),
    mpatches.Patch(color=[200/255,100/255,100/255], label='Pred only (FP)'),
    mpatches.Patch(color=[200/255,200/255,100/255], label='교집합 (TP)'),
]
axes[0].legend(handles=patches, fontsize=8, loc='lower right')
axes[0].axis('off')

# IoU vs Dice 이론 곡선
iou_vals  = np.linspace(0, 1, 200)
dice_vals = 2 * iou_vals / (1 + iou_vals)
axes[1].plot(iou_vals, dice_vals, 'b-', linewidth=2, label='Dice = 2·IoU/(1+IoU)')
axes[1].plot([0,1], [0,1], 'k--', alpha=0.4, label='y=x')
axes[1].scatter([iou], [dice], color='red', s=100, zorder=5, label=f'현재 (IoU={iou:.3f}, Dice={dice:.3f})')
axes[1].set_xlabel('IoU'); axes[1].set_ylabel('Dice')
axes[1].set_title('IoU vs Dice 관계')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

# Overlap 비율별 IoU/Dice
overlaps = np.linspace(0, 1, 50)
gt_size  = 3600
pd_size  = 2500
iou_curve  = []
dice_curve = []
for ov in overlaps:
    inter = ov * min(gt_size, pd_size)
    union = gt_size + pd_size - inter
    iou_curve.append(inter / union)
    dice_curve.append(2*inter / (gt_size + pd_size))
axes[2].plot(overlaps, iou_curve,  'b-', linewidth=2, label='IoU')
axes[2].plot(overlaps, dice_curve, 'r-', linewidth=2, label='Dice')
axes[2].set_xlabel('Overlap 비율 (inter/min_size)')
axes[2].set_ylabel('Score')
axes[2].set_title('Overlap 비율별 IoU/Dice')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('Segmentation 평가 지표 — IoU & Dice', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. SAM2 아키텍처 원리

### Segment Anything Model (SAM, Meta 2023)

SAM은 세 가지 모듈로 구성됩니다:

```
입력 이미지
    │
    ▼
┌─────────────────────────┐
│  Image Encoder (ViT-H)  │  → 1024-dim 이미지 임베딩 (한 번만 계산, 재사용)
└─────────────────────────┘
    │
    ├─────────────────────────┐
    ▼                         ▼
┌──────────────────┐   ┌──────────────────────┐
│ Prompt Encoder   │   │  Image Embedding      │
│  Point / Box /   │   │  (64×64 feature map)  │
│  Mask / Text     │   └──────────────────────┘
└──────────────────┘
    │                         │
    └───────────┬─────────────┘
                ▼
┌───────────────────────────┐
│   Mask Decoder            │
│   (Transformer 2-layer)   │
│   → 3개의 마스크 후보 출력  │
│   → IoU 예측 (confidence) │
└───────────────────────────┘
```

### SAM2 (2024) 추가 기능
- **Hiera 백본** (ViT 대신 계층형 ViT → 속도 향상)
- **Memory Attention** — 비디오 시퀀스에서 이전 프레임 마스크 기억
- **Streaming Memory** — 실시간 비디오 추적 가능

### 왜 SAM이 강력한가?
```
기존 세그멘테이션: 클래스별 fine-tuning 필요 (DeepLab, Mask R-CNN)
SAM             : Zero-shot — 어떤 객체도 프롬프트만으로 세그멘테이션
```

In [ ]:
# ── SAM2 아키텍처 개념 시각화 ──
fig, ax = plt.subplots(1, 1, figsize=(14, 6))
ax.set_xlim(0, 14); ax.set_ylim(0, 6); ax.axis('off')

def draw_box(ax, x, y, w, h, color, label, fontsize=9):
    rect = mpatches.FancyBboxPatch((x, y), w, h,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x+w/2, y+h/2, label, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', wrap=True)

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

# 박스들
draw_box(ax, 0.3, 4.2, 2.4, 1.2, '#AED6F1', '입력 이미지\n(H×W×3)')
draw_box(ax, 0.3, 2.0, 2.4, 1.4, '#A9DFBF', 'Image Encoder\n(ViT-H / Hiera)\n→ 64×64 임베딩', 8)
draw_box(ax, 3.5, 4.2, 2.4, 1.2, '#F9E79F', 'Prompt Encoder\n(Point / Box / Mask)')
draw_box(ax, 3.5, 2.0, 2.4, 1.4, '#F0B27A', 'Prompt Embedding\n(Sparse+Dense)', 8)
draw_box(ax, 7.0, 2.5, 3.0, 1.2, '#D7BDE2', 'Mask Decoder\n(Transformer 2-layer)\n→ 3개 마스크 후보', 8)
draw_box(ax, 11.0, 3.5, 2.5, 1.0, '#F1948A', '출력 마스크\n+ IoU 점수')
draw_box(ax, 11.0, 1.5, 2.5, 1.2, '#85C1E9', 'Memory\nModule\n(SAM2 전용)', 8)

# 화살표
draw_arrow(ax, 1.5, 4.2, 1.5, 3.4)
draw_arrow(ax, 4.7, 4.2, 4.7, 3.4)
draw_arrow(ax, 2.7, 2.7, 7.0, 3.0)
draw_arrow(ax, 5.9, 2.7, 7.0, 3.0)
draw_arrow(ax, 10.0, 3.1, 11.0, 4.0)
draw_arrow(ax, 10.0, 3.1, 11.0, 2.2)

ax.text(7.0, 5.7, 'SAM2 아키텍처', fontsize=14, fontweight='bold', ha='center')
ax.text(1.5, 1.6, '① 이미지당 1회만\n   임베딩 계산', fontsize=7, ha='center', color='gray')
ax.text(4.7, 1.6, '② 프롬프트마다\n   즉시 계산', fontsize=7, ha='center', color='gray')
ax.text(8.5, 1.6, '③ 경량 디코더\n   (실시간 가능)', fontsize=7, ha='center', color='gray')
plt.tight_layout()
plt.show()

## 3. SAM2 모델 로드 + Point Prompt 추론

Ultralytics를 통해 SAM2 사용 (자동 다운로드).

### 프롬프트 방식 3가지
| 방식 | 입력 | 장점 | 단점 |
|------|------|------|------|
| **Point** | 클릭 좌표 + fg/bg 라벨 | 직관적, 빠름 | 애매한 경우 불정확 |
| **Box** | 바운딩 박스 | Detection과 연동 용이 | 박스 정확도에 의존 |
| **Automatic** | 없음 | 전체 객체 자동 분할 | 느림 |

In [ ]:
from ultralytics import SAM

# SAM2 소형 모델 (자동 다운로드 ~40MB)
sam_model = SAM('sam2.1_b.pt')
print('SAM2 모델 로드 완료')
print(f'파라미터 수: {sum(p.numel() for p in sam_model.model.parameters()):,}')

# 테스트 이미지 선택 (COCO128)
img_files = sorted(COCO_IMG_DIR.glob('*.jpg'))[:12]
test_img_path = str(img_files[2])   # 세 번째 이미지
test_img = cv2.imread(test_img_path)
test_img = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)
H_img, W_img = test_img.shape[:2]
print(f'\n테스트 이미지: {Path(test_img_path).name}  ({W_img}×{H_img})')

In [ ]:
# ── Point Prompt: 이미지 중앙 + 1/4 지점 ──
cx, cy = W_img // 2, H_img // 2
point_coords = [[cx, cy]]
point_labels = [1]  # 1=foreground, 0=background

results_point = sam_model(
    test_img_path,
    points=point_coords,
    labels=point_labels,
    verbose=False
)

# 마스크 추출
masks_point = []
if results_point[0].masks is not None:
    masks_point = results_point[0].masks.data.cpu().numpy()  # (N, H, W)

print(f'Point Prompt → 생성된 마스크 수: {len(masks_point)}')
if len(masks_point) > 0:
    m = masks_point[0]
    print(f'  마스크 shape: {m.shape}, 픽셀 커버리지: {m.mean()*100:.1f}%')

# ── 시각화 ──
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(test_img)
axes[0].scatter([cx], [cy], c='red', s=200, zorder=5, marker='*')
axes[0].set_title('원본 이미지 + Point Prompt', fontsize=11)
axes[0].axis('off')

if len(masks_point) > 0:
    overlay = test_img.copy()
    mask_bool = masks_point[0].astype(bool)
    overlay[mask_bool] = overlay[mask_bool] * 0.5 + np.array([0, 150, 255]) * 0.5
    axes[1].imshow(overlay.astype(np.uint8))
    axes[1].scatter([cx], [cy], c='red', s=200, zorder=5, marker='*')
    axes[1].set_title(f'SAM2 Point 세그멘테이션\n(커버리지: {masks_point[0].mean()*100:.1f}%)', fontsize=11)
else:
    axes[1].imshow(test_img); axes[1].set_title('마스크 없음')
axes[1].axis('off')

if len(masks_point) > 0:
    axes[2].imshow(masks_point[0], cmap='Blues')
    axes[2].set_title('마스크 이진 맵', fontsize=11)
else:
    axes[2].imshow(np.zeros_like(test_img[:,:,0]))
axes[2].axis('off')

plt.suptitle('SAM2 — Point Prompt 추론', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Box Prompt — Detection 결과와 연동

YOLO 레이블(바운딩 박스)을 그대로 SAM2 Box Prompt로 사용 →
Detection → Segmentation 파이프라인의 핵심 연결 고리.

In [ ]:
def load_yolo_labels(img_path: Path, lbl_dir: Path, img_w: int, img_h: int):
    """YOLO 형식 레이블 → 절대 좌표 bbox 리스트 반환"""
    lbl_path = lbl_dir / (img_path.stem + '.txt')
    boxes = []
    if not lbl_path.exists():
        return boxes
    with open(lbl_path) as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            if len(parts) < 5:
                continue
            cls_id, cx_n, cy_n, w_n, h_n = int(parts[0]), *parts[1:5]
            x1 = int((cx_n - w_n/2) * img_w)
            y1 = int((cy_n - h_n/2) * img_h)
            x2 = int((cx_n + w_n/2) * img_w)
            y2 = int((cy_n + h_n/2) * img_h)
            boxes.append({'cls': cls_id, 'box': [x1,y1,x2,y2]})
    return boxes


# COCO 클래스 이름 (자율주행 관련만 하이라이트)
COCO_NAMES = [
    'person','bicycle','car','motorcycle','airplane','bus','train','truck',
    'boat','traffic light','fire hydrant','stop sign','parking meter','bench',
    'bird','cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe',
    'backpack','umbrella','handbag','tie','suitcase','frisbee','skis','snowboard',
    'sports ball','kite','baseball bat','baseball glove','skateboard','surfboard',
    'tennis racket','bottle','wine glass','cup','fork','knife','spoon','bowl',
    'banana','apple','sandwich','orange','broccoli','carrot','hot dog','pizza',
    'donut','cake','chair','couch','potted plant','bed','dining table','toilet',
    'tv','laptop','mouse','remote','keyboard','cell phone','microwave','oven',
    'toaster','sink','refrigerator','book','clock','vase','scissors','teddy bear',
    'hair drier','toothbrush'
]
AV_CLASSES = {0:'person',1:'bicycle',2:'car',3:'motorcycle',5:'bus',7:'truck',9:'traffic light',11:'stop sign'}

# 박스 있는 이미지 찾기
target_img_path = None
target_boxes    = None
for img_f in img_files:
    boxes = load_yolo_labels(img_f, COCO_LBL_DIR, *cv2.imread(str(img_f)).shape[1::-1])
    if len(boxes) >= 2:
        img_tmp = cv2.imread(str(img_f))
        h, w = img_tmp.shape[:2]
        target_img_path = img_f
        target_boxes    = load_yolo_labels(img_f, COCO_LBL_DIR, w, h)
        break

print(f'선택 이미지: {target_img_path.name}, bbox 수: {len(target_boxes)}')
for b in target_boxes:
    cls_name = COCO_NAMES[b["cls"]] if b["cls"] < len(COCO_NAMES) else str(b["cls"])
    print(f'  cls={cls_name}, box={b["box"]}')

In [ ]:
# ── Box Prompt → SAM2 추론 ──
target_img = cv2.cvtColor(cv2.imread(str(target_img_path)), cv2.COLOR_BGR2RGB)
H_t, W_t = target_img.shape[:2]

bboxes = [b['box'] for b in target_boxes]

results_box = sam_model(
    str(target_img_path),
    bboxes=bboxes,
    verbose=False
)

masks_box = []
if results_box[0].masks is not None:
    masks_box = results_box[0].masks.data.cpu().numpy()

print(f'Box Prompt 수: {len(bboxes)}, 생성 마스크 수: {len(masks_box)}')

# ── 시각화 ──
colors_mask = [
    [255, 100, 100], [100, 255, 100], [100, 100, 255],
    [255, 255, 100], [255, 100, 255], [100, 255, 255]
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 원본 + bbox
vis_orig = target_img.copy()
for i, b in enumerate(target_boxes):
    x1,y1,x2,y2 = b['box']
    cls_name = COCO_NAMES[b['cls']] if b['cls'] < len(COCO_NAMES) else str(b['cls'])
    color = colors_mask[i % len(colors_mask)]
    cv2.rectangle(vis_orig, (x1,y1), (x2,y2), color, 2)
    cv2.putText(vis_orig, cls_name, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
axes[0].imshow(vis_orig)
axes[0].set_title('YOLO 바운딩 박스 (Box Prompt)', fontsize=11)
axes[0].axis('off')

# SAM2 마스크 오버레이
vis_mask = target_img.copy().astype(np.float32)
for i, mask in enumerate(masks_box):
    color = np.array(colors_mask[i % len(colors_mask)], dtype=np.float32)
    mask_bool = mask.astype(bool)
    vis_mask[mask_bool] = vis_mask[mask_bool] * 0.45 + color * 0.55

for i, b in enumerate(target_boxes):
    x1,y1,x2,y2 = b['box']
    cls_name = COCO_NAMES[b['cls']] if b['cls'] < len(COCO_NAMES) else str(b['cls'])
    color = colors_mask[i % len(colors_mask)]
    cv2.rectangle(vis_mask, (x1,y1), (x2,y2), color, 1)

axes[1].imshow(vis_mask.astype(np.uint8))
axes[1].set_title(f'SAM2 Box Prompt 세그멘테이션\n({len(masks_box)}개 마스크)', fontsize=11)
axes[1].axis('off')

plt.suptitle('Detection(Box) → SAM2 세그멘테이션 파이프라인', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 마스크별 IoU/Dice 출력
print('\n[마스크 통계]')
for i, mask in enumerate(masks_box):
    cls_name = COCO_NAMES[target_boxes[i]['cls']] if i < len(target_boxes) and target_boxes[i]['cls'] < len(COCO_NAMES) else f'obj{i}'
    coverage = mask.mean() * 100
    print(f'  [{cls_name}] 픽셀 커버리지: {coverage:.1f}%  ({mask.sum():,}픽셀)')

## 5. mIoU 평가 파이프라인

### 전략
세그멘테이션 GT 마스크가 없으므로:
1. **Box Prompt** → SAM2 마스크 → **Pseudo-GT** (기준)
2. **Point Prompt** (박스 중심점) → SAM2 마스크 → **예측**
3. 두 마스크간 IoU 계산 → **Point vs Box 일치도 측정**

이는 실제 파이프라인에서 "Detection으로 구한 박스 기반 마스크" vs "단순 클릭 마스크"의 품질 차이를 정량화하는 방법과 동일.

In [ ]:
def evaluate_point_vs_box_iou(img_path: Path, lbl_dir: Path, model) -> list:
    """
    단일 이미지에 대해:
    - Box prompt → 마스크 (pseudo-GT)
    - 박스 중심 Point prompt → 마스크 (pred)
    - mask_iou 계산
    """
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    boxes_info = load_yolo_labels(img_path, lbl_dir, w, h)
    if not boxes_info:
        return []

    bboxes = [b['box'] for b in boxes_info]

    # Box prompt
    res_box = model(str(img_path), bboxes=bboxes, verbose=False)
    if res_box[0].masks is None:
        return []
    masks_b = res_box[0].masks.data.cpu().numpy()

    results = []
    for i, (b_info, mask_b) in enumerate(zip(boxes_info, masks_b)):
        x1,y1,x2,y2 = b_info['box']
        cx_pt = (x1+x2)//2
        cy_pt = (y1+y2)//2

        # Point prompt (박스 중심)
        res_pt = model(str(img_path), points=[[cx_pt,cy_pt]], labels=[1], verbose=False)
        if res_pt[0].masks is None:
            continue
        mask_p = res_pt[0].masks.data.cpu().numpy()[0]

        iou  = mask_iou(mask_p, mask_b)
        dice = mask_dice(mask_p, mask_b)
        cls_name = COCO_NAMES[b_info['cls']] if b_info['cls'] < len(COCO_NAMES) else str(b_info['cls'])
        results.append({'cls': cls_name, 'iou': iou, 'dice': dice,
                        'coverage_box': mask_b.mean(), 'coverage_pt': mask_p.mean()})
    return results


# ── 다중 이미지 평가 (상위 10장) ──
print('Box vs Point Prompt 마스크 일치도 평가 중...')
all_results = []
eval_imgs = []
for img_f in img_files[:10]:
    img_tmp = cv2.imread(str(img_f))
    h, w = img_tmp.shape[:2]
    boxes = load_yolo_labels(img_f, COCO_LBL_DIR, w, h)
    if boxes:
        eval_imgs.append(img_f)

for img_f in eval_imgs[:6]:
    res = evaluate_point_vs_box_iou(img_f, COCO_LBL_DIR, sam_model)
    all_results.extend(res)
    print(f'  {img_f.name}: {len(res)}개 객체 평가')

print(f'\n총 평가 객체 수: {len(all_results)}')
if all_results:
    mean_iou  = np.mean([r['iou']  for r in all_results])
    mean_dice = np.mean([r['dice'] for r in all_results])
    print(f'평균 IoU  (Point vs Box): {mean_iou:.4f}')
    print(f'평균 Dice (Point vs Box): {mean_dice:.4f}')

In [ ]:
# ── 클래스별 mIoU 집계 및 시각화 ──
from collections import defaultdict

cls_iou = defaultdict(list)
for r in all_results:
    cls_iou[r['cls']].append(r['iou'])

cls_mean_iou = {cls: np.mean(vals) for cls, vals in cls_iou.items()}
cls_sorted    = sorted(cls_mean_iou.items(), key=lambda x: -x[1])

if cls_sorted:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 클래스별 mIoU 바 차트
    classes = [c for c,_ in cls_sorted]
    values  = [v for _,v in cls_sorted]
    av_mask = [c in AV_CLASSES.values() for c in classes]
    bar_colors = ['#E74C3C' if m else '#85C1E9' for m in av_mask]
    bars = axes[0].bar(range(len(classes)), values, color=bar_colors)
    axes[0].set_xticks(range(len(classes)))
    axes[0].set_xticklabels(classes, rotation=45, ha='right', fontsize=9)
    axes[0].axhline(y=np.mean(values), color='orange', linestyle='--', label=f'mIoU={np.mean(values):.3f}')
    axes[0].set_ylim(0, 1.1)
    axes[0].set_ylabel('IoU (Point vs Box)')
    axes[0].set_title('클래스별 Point-Box IoU 일치도')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')
    legend_patches = [
        mpatches.Patch(color='#E74C3C', label='자율주행 핵심 클래스'),
        mpatches.Patch(color='#85C1E9', label='일반 클래스')
    ]
    axes[0].legend(handles=legend_patches + [plt.Line2D([0],[0],color='orange',linestyle='--',label=f'평균 mIoU={np.mean(values):.3f}')])

    # IoU 분포 히스토그램
    all_ious = [r['iou'] for r in all_results]
    axes[1].hist(all_ious, bins=20, color='#85C1E9', edgecolor='black', alpha=0.8)
    axes[1].axvline(x=np.mean(all_ious), color='red', linestyle='--', label=f'평균={np.mean(all_ious):.3f}')
    axes[1].axvline(x=np.median(all_ious), color='orange', linestyle='--', label=f'중앙값={np.median(all_ious):.3f}')
    axes[1].set_xlabel('IoU'); axes[1].set_ylabel('빈도')
    axes[1].set_title('IoU 분포 히스토그램')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.suptitle('mIoU 평가 파이프라인 결과 (Point vs Box 일치도)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('평가 결과 없음')

## 6. 자율주행 응용 — 클래스별 세그멘테이션 분석

### 자율주행에서 세그멘테이션이 필요한 이유
```
Detection (bbox)  : 객체 위치·크기만 알 수 있음
Segmentation (mask): 객체의 정확한 형상·경계 파악 → 더 정밀한 경로 계획 가능
```

| 응용 | Detection | Segmentation |
|------|-----------|-------------|
| 차선 인식 | ❌ (선 형태 bbox 부적합) | ✅ (픽셀 단위 마스크) |
| 보행자 영역 | bbox 근사 | 정확한 실루엣 |
| 자유 주행 공간 | 간접 추론 | 직접 픽셀 추출 |
| HD 맵 생성 | ❌ | ✅ |

In [ ]:
# ── 자율주행 핵심 클래스 세그멘테이션 분석 ──
av_results = [r for r in all_results if r['cls'] in AV_CLASSES.values()]

print('=== 자율주행 핵심 클래스 세그멘테이션 성능 ===')
print(f'{'클래스':<15} {'count':>5} {'mean IoU':>10} {'mean Dice':>11} {'커버리지(Box)':>14}')
print('-' * 60)

av_cls_iou = defaultdict(list)
av_cls_dice = defaultdict(list)
av_cls_cov  = defaultdict(list)
for r in av_results:
    av_cls_iou[r['cls']].append(r['iou'])
    av_cls_dice[r['cls']].append(r['dice'])
    av_cls_cov[r['cls']].append(r['coverage_box'])

for cls in sorted(av_cls_iou.keys()):
    n    = len(av_cls_iou[cls])
    miou = np.mean(av_cls_iou[cls])
    dice = np.mean(av_cls_dice[cls])
    cov  = np.mean(av_cls_cov[cls]) * 100
    print(f'{cls:<15} {n:>5} {miou:>10.4f} {dice:>11.4f} {cov:>13.1f}%')

print('-' * 60)
if av_results:
    print(f'{'[전체 평균]':<15} {len(av_results):>5} {np.mean([r["iou"] for r in av_results]):>10.4f} {np.mean([r["dice"] for r in av_results]):>11.4f}')
else:
    print('[자율주행 클래스 객체가 현재 평가 이미지에 없습니다]')

print()
print('[해석]')
print('- IoU 높음(>0.7): Point 클릭만으로도 Box와 동등한 마스크 생성 → SAM2 일반화 우수')
print('- IoU 낮음(<0.5): 프롬프트에 민감한 객체 → 자율주행에서 Box 기반 접근 필수')
print('- 커버리지 작음(<5%): 작은 객체 (먼 거리 차량 등) → 고해상도 입력 필요')

In [ ]:
# ── 여러 이미지 멀티-클래스 세그멘테이션 시각화 ──
n_show = min(4, len(eval_imgs))
fig, axes = plt.subplots(2, n_show, figsize=(4*n_show, 8))
if n_show == 1:
    axes = axes.reshape(2, 1)

for col, img_f in enumerate(eval_imgs[:n_show]):
    img = cv2.cvtColor(cv2.imread(str(img_f)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    boxes_info = load_yolo_labels(img_f, COCO_LBL_DIR, w, h)
    bboxes = [b['box'] for b in boxes_info]

    # 원본
    axes[0][col].imshow(img)
    axes[0][col].set_title(img_f.name[:15], fontsize=8)
    axes[0][col].axis('off')

    # SAM2 마스크
    if bboxes:
        res = sam_model(str(img_f), bboxes=bboxes, verbose=False)
        vis = img.copy().astype(np.float32)
        if res[0].masks is not None:
            for i, mask in enumerate(res[0].masks.data.cpu().numpy()):
                color = np.array(colors_mask[i % len(colors_mask)], dtype=np.float32)
                vis[mask.astype(bool)] = vis[mask.astype(bool)] * 0.4 + color * 0.6
            axes[1][col].imshow(vis.astype(np.uint8))
            axes[1][col].set_title(f'SAM2 ({res[0].masks.shape[0]}개)', fontsize=8)
        else:
            axes[1][col].imshow(img)
            axes[1][col].set_title('마스크 없음', fontsize=8)
    else:
        axes[1][col].imshow(img)
        axes[1][col].set_title('레이블 없음', fontsize=8)
    axes[1][col].axis('off')

axes[0][0].set_ylabel('원본', fontsize=10)
axes[1][0].set_ylabel('SAM2 마스크', fontsize=10)
plt.suptitle('COCO128 멀티-이미지 세그멘테이션 (Box Prompt)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. SAM vs SAM2 비교 + 자율주행 파이프라인 포지셔닝

### SAM vs SAM2 핵심 차이
| 항목 | SAM (2023) | SAM2 (2024) |
|------|-----------|------------|
| 백본 | ViT-H (300M) | Hiera-L (224M) |
| 속도 | ~50ms/frame | ~8ms/frame |
| 비디오 | ❌ | ✅ (Memory Module) |
| 실시간 | 어려움 | 가능 |
| 자율주행 적합 | 오프라인 분석 | 온라인 추적 |

### 자율주행 Perception 파이프라인에서의 위치
```
카메라 입력
    │
    ├─→ [YOLOv8 Detection]  → bbox + class
    │         │
    │         └─→ [SAM2 Box Prompt] → 정밀 마스크
    │                   │
    │                   └─→ [ByteTrack] → 마스크 추적 (Phase 2)
    │
    ├─→ [DepthAnythingV2]   → 깊이 맵 (Phase 1 Depth)
    │
    └─→ [IPM/BEV]          → 지면 좌표 (Phase 3)
```

### 한계 및 개선 방향
```
현재 한계:
  - GT 마스크 없어 절대적 mIoU 측정 불가 (COCO128은 bbox만 제공)
  - 실시간 성능 미측정

개선 방향 (Phase 4 CARLA):
  - CARLA 시뮬레이터 → semantic segmentation GT 자동 생성
  - 실제 mIoU@CARLA 측정 가능
  - SAM2 파인튜닝 → 자율주행 도메인 특화
```

## 8. 최종 결과 요약 및 Phase 1 Depth 연결

In [ ]:
print('=' * 65)
print('Phase 1 Segmentation 파이프라인 최종 결과')
print('=' * 65)
print()
print('[구현 완료]')
print('  1. Mask IoU     — 픽셀 단위 교집합/합집합 직접 구현')
print('  2. Dice Coeff   — 2·TP/(2TP+FP+FN) 직접 구현')
print('  3. SAM2 추론    — Point / Box Prompt 비교')
print('  4. mIoU 파이프라인 — 다중 이미지 클래스별 집계')
print('  5. 자율주행 응용 — 핵심 클래스 분석')
print()
print('[평가 결과]')
if all_results:
    print(f'  총 평가 객체    : {len(all_results)}개')
    print(f'  평균 mIoU       : {np.mean([r["iou"] for r in all_results]):.4f}  (Point vs Box 일치도)')
    print(f'  평균 Dice       : {np.mean([r["dice"] for r in all_results]):.4f}')
    best = max(cls_mean_iou.items(), key=lambda x:x[1]) if cls_mean_iou else None
    if best: print(f'  최고 IoU 클래스 : {best[0]} ({best[1]:.4f})')
print()
print('[Detection → Segmentation 연결 고리]')
print('  Phase 1 Det    bbox + IoU(box)   → mAP')
print('  Phase 1 Seg    mask + IoU(mask)  → mIoU')
print('  Phase 2 Track  mask + OKS track  → HOTA')
print('  Phase 4 CARLA  GT mask 자동생성  → 절대적 mIoU 측정')
print()
print('[Phase 1 Depth 연결]')
print('  Segmentation mask + Depth map → 3D 객체 영역 추정')
print('  예: 보행자 마스크 내 평균 깊이 → 보행자까지 거리 계산')
print('=' * 65)